# 🛒 Supermarket Sales Analysis
**Objective:** Analyze supermarket sales data to uncover trends in revenue, products, customer behavior and branch performance.

**Author:** Nimitha BC | **Date:** June 2026

---

## 📌 Table of Contents
1. Import Libraries
2. Load Dataset
3. Exploratory Data Analysis
4. Sales by Branch
5. Sales by Product Line
6. Customer Analysis
7. Payment Method Analysis
8. Time Trend Analysis
9. Rating Analysis
10. Correlation Heatmap
11. Conclusion

## 1️⃣ Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

GREEN  = '#2ecc71'
DARK   = '#1a1a2e'
RED    = '#e74c3c'
BLUE   = '#3498db'
ORANGE = '#f39c12'
PURPLE = '#9b59b6'

print('✅ All libraries imported successfully!')

## 2️⃣ Load Dataset

In [ ]:
url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/supermarket_sales.csv'
try:
    df = pd.read_csv(url)
    print(f'✅ Loaded from URL! Shape: {df.shape}')
except:
    import glob
    files = glob.glob(r'C:\Users\nimit\Downloads\**\*.csv', recursive=True)
    sm_files = [f for f in files if 'super' in f.lower() or 'sales' in f.lower()]
    if sm_files:
        df = pd.read_csv(sm_files[0])
        print(f'✅ Loaded from file: {sm_files[0]}')
    else:
        df = pd.read_csv(files[0])
        print(f'✅ Loaded: {files[0]}')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3️⃣ Exploratory Data Analysis

In [ ]:
print('🔍 Missing Values:')
print(df.isnull().sum())

# Fix date column
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'])
    df['Month'] = df['Date'].dt.month_name()
    df['Day'] = df['Date'].dt.day_name()
    df['Week'] = df['Date'].dt.isocalendar().week.astype(int)

# Fix time column
if 'Time' in df.columns:
    df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M').dt.hour

total_sales = df['Total'].sum() if 'Total' in df.columns else 0
print(f'\n📊 Total Revenue: ${total_sales:,.2f}')
print(f'🛒 Total Transactions: {len(df)}')
print(f'⭐ Average Rating: {df["Rating"].mean():.2f}' if 'Rating' in df.columns else '')
print(f'💰 Average Sale: ${df["Total"].mean():.2f}' if 'Total' in df.columns else '')

## 4️⃣ Sales by Branch

In [ ]:
# Plot 1: Sales by Branch
if 'Branch' in df.columns and 'Total' in df.columns:
    branch_sales = df.groupby('Branch')['Total'].sum().sort_values(ascending=False)
    branch_count = df.groupby('Branch')['Total'].count()

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor(DARK)

    axes[0].set_facecolor(DARK)
    bars = axes[0].bar(branch_sales.index, branch_sales.values,
                       color=[GREEN, BLUE, ORANGE], edgecolor='black')
    axes[0].set_title('Total Revenue by Branch', color='white', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Total Revenue ($)', color='white')
    axes[0].set_xlabel('Branch', color='white')
    axes[0].tick_params(colors='white')
    for bar, val in zip(bars, branch_sales.values):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
                     f'${val:,.0f}', ha='center', color='white', fontweight='bold')

    axes[1].set_facecolor(DARK)
    axes[1].pie(branch_sales.values, labels=branch_sales.index,
                colors=[GREEN, BLUE, ORANGE], autopct='%1.1f%%',
                startangle=90, textprops={'color': 'white'})
    axes[1].set_title('Revenue Share by Branch', color='white', fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig('plot1_branch_sales.png', dpi=150, bbox_inches='tight', facecolor=DARK)
    plt.show()
    print('✅ Plot 1 saved!')
    print(branch_sales)

## 5️⃣ Sales by Product Line

In [ ]:
# Plot 2: Sales by Product Line
prod_col = 'Product line' if 'Product line' in df.columns else 'Product Line'
if prod_col in df.columns:
    prod_sales = df.groupby(prod_col)['Total'].sum().sort_values(ascending=True)
    prod_qty = df.groupby(prod_col)['Quantity'].sum().sort_values(ascending=True) if 'Quantity' in df.columns else None

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.patch.set_facecolor(DARK)

    axes[0].set_facecolor(DARK)
    colors_p = [GREEN, BLUE, ORANGE, RED, PURPLE, '#1abc9c']
    bars = axes[0].barh(prod_sales.index, prod_sales.values, color=colors_p)
    axes[0].set_title('Revenue by Product Line', color='white', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Total Revenue ($)', color='white')
    axes[0].tick_params(colors='white')
    for bar, val in zip(bars, prod_sales.values):
        axes[0].text(bar.get_width()+100, bar.get_y()+bar.get_height()/2,
                     f'${val:,.0f}', va='center', color='white', fontsize=9)

    if prod_qty is not None:
        axes[1].set_facecolor(DARK)
        bars2 = axes[1].barh(prod_qty.index, prod_qty.values, color=colors_p)
        axes[1].set_title('Quantity Sold by Product Line', color='white', fontsize=13, fontweight='bold')
        axes[1].set_xlabel('Total Quantity Sold', color='white')
        axes[1].tick_params(colors='white')
        for bar, val in zip(bars2, prod_qty.values):
            axes[1].text(bar.get_width()+5, bar.get_y()+bar.get_height()/2,
                         str(val), va='center', color='white', fontsize=9)

    plt.tight_layout()
    plt.savefig('plot2_product_line.png', dpi=150, bbox_inches='tight', facecolor=DARK)
    plt.show()
    print('✅ Plot 2 saved!')

## 6️⃣ Customer Analysis

In [ ]:
# Plot 3: Customer Type and Gender
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor(DARK)

if 'Customer type' in df.columns:
    axes[0].set_facecolor(DARK)
    cust = df.groupby('Customer type')['Total'].sum()
    bars = axes[0].bar(cust.index, cust.values, color=[GREEN, BLUE], edgecolor='black')
    axes[0].set_title('Revenue by Customer Type', color='white', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('Total Revenue ($)', color='white')
    axes[0].tick_params(colors='white')
    for bar, val in zip(bars, cust.values):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                     f'${val:,.0f}', ha='center', color='white', fontweight='bold')

if 'Gender' in df.columns:
    axes[1].set_facecolor(DARK)
    gender = df.groupby('Gender')['Total'].sum()
    axes[1].pie(gender.values, labels=gender.index,
                colors=[PURPLE, ORANGE], autopct='%1.1f%%',
                startangle=90, textprops={'color': 'white'})
    axes[1].set_title('Revenue by Gender', color='white', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('plot3_customer_analysis.png', dpi=150, bbox_inches='tight', facecolor=DARK)
plt.show()
print('✅ Plot 3 saved!')

## 7️⃣ Payment Method Analysis

In [ ]:
# Plot 4: Payment Methods
if 'Payment' in df.columns:
    payment = df['Payment'].value_counts()
    payment_rev = df.groupby('Payment')['Total'].sum()

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor(DARK)

    axes[0].set_facecolor(DARK)
    axes[0].pie(payment.values, labels=payment.index,
                colors=[GREEN, BLUE, ORANGE], autopct='%1.1f%%',
                startangle=90, textprops={'color': 'white'})
    axes[0].set_title('Payment Method Usage', color='white', fontsize=13, fontweight='bold')

    axes[1].set_facecolor(DARK)
    bars = axes[1].bar(payment_rev.index, payment_rev.values,
                       color=[GREEN, BLUE, ORANGE], edgecolor='black')
    axes[1].set_title('Revenue by Payment Method', color='white', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Total Revenue ($)', color='white')
    axes[1].tick_params(colors='white')
    for bar, val in zip(bars, payment_rev.values):
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                     f'${val:,.0f}', ha='center', color='white', fontweight='bold')

    plt.tight_layout()
    plt.savefig('plot4_payment.png', dpi=150, bbox_inches='tight', facecolor=DARK)
    plt.show()
    print('✅ Plot 4 saved!')

## 8️⃣ Time Trend Analysis

In [ ]:
# Plot 5: Sales by Month and Day
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor(DARK)

if 'Month' in df.columns:
    month_order = ['January','February','March','April','May','June',
                   'July','August','September','October','November','December']
    month_sales = df.groupby('Month')['Total'].sum().reindex(
        [m for m in month_order if m in df['Month'].unique()])
    axes[0].set_facecolor(DARK)
    axes[0].bar(month_sales.index, month_sales.values, color=GREEN, edgecolor='black')
    axes[0].set_title('Revenue by Month', color='white', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('Total Revenue ($)', color='white')
    axes[0].tick_params(colors='white')
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30, ha='right', color='white')

if 'Day' in df.columns:
    day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    day_sales = df.groupby('Day')['Total'].sum().reindex(
        [d for d in day_order if d in df['Day'].unique()])
    axes[1].set_facecolor(DARK)
    axes[1].bar(day_sales.index, day_sales.values, color=ORANGE, edgecolor='black')
    axes[1].set_title('Revenue by Day of Week', color='white', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Total Revenue ($)', color='white')
    axes[1].tick_params(colors='white')
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right', color='white')

plt.tight_layout()
plt.savefig('plot5_time_trend.png', dpi=150, bbox_inches='tight', facecolor=DARK)
plt.show()
print('✅ Plot 5 saved!')

In [ ]:
# Plot 6: Sales by Hour
if 'Hour' in df.columns:
    hourly = df.groupby('Hour')['Total'].sum()

    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor(DARK)
    ax.set_facecolor(DARK)

    ax.plot(hourly.index, hourly.values, color=BLUE,
            marker='o', linewidth=2.5, markersize=8)
    ax.fill_between(hourly.index, hourly.values, alpha=0.3, color=BLUE)
    ax.set_title('Revenue by Hour of Day', color='white', fontsize=14, fontweight='bold')
    ax.set_xlabel('Hour of Day', color='white')
    ax.set_ylabel('Total Revenue ($)', color='white')
    ax.tick_params(colors='white')
    ax.set_xticks(hourly.index)

    plt.tight_layout()
    plt.savefig('plot6_hourly_sales.png', dpi=150, bbox_inches='tight', facecolor=DARK)
    plt.show()
    print('✅ Plot 6 saved!')

## 9️⃣ Rating Analysis

In [ ]:
# Plot 7: Rating Analysis
if 'Rating' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor(DARK)

    axes[0].set_facecolor(DARK)
    axes[0].hist(df['Rating'], bins=20, color=ORANGE, edgecolor='black', alpha=0.85)
    axes[0].axvline(df['Rating'].mean(), color=RED, linestyle='--',
                    linewidth=2, label=f'Mean: {df["Rating"].mean():.2f}')
    axes[0].set_title('Customer Rating Distribution', color='white', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Rating', color='white')
    axes[0].set_ylabel('Count', color='white')
    axes[0].tick_params(colors='white')
    axes[0].legend(facecolor=DARK, labelcolor='white')

    if prod_col in df.columns:
        axes[1].set_facecolor(DARK)
        prod_rating = df.groupby(prod_col)['Rating'].mean().sort_values()
        bars = axes[1].barh(prod_rating.index, prod_rating.values, color=PURPLE)
        axes[1].set_title('Avg Rating by Product Line', color='white', fontsize=13, fontweight='bold')
        axes[1].set_xlabel('Average Rating', color='white')
        axes[1].tick_params(colors='white')
        axes[1].set_xlim(0, 10)
        for bar, val in zip(bars, prod_rating.values):
            axes[1].text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
                         f'{val:.2f}', va='center', color='white')

    plt.tight_layout()
    plt.savefig('plot7_rating_analysis.png', dpi=150, bbox_inches='tight', facecolor=DARK)
    plt.show()
    print('✅ Plot 7 saved!')

## 🔟 Correlation Heatmap

In [ ]:
# Plot 8: Correlation Heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
useful_cols = [c for c in numeric_cols if df[c].notna().sum() > 50]

if len(useful_cols) >= 2:
    fig, ax = plt.subplots(figsize=(11, 8))
    fig.patch.set_facecolor(DARK)
    sns.heatmap(df[useful_cols].corr(), annot=True, fmt='.2f',
                cmap='RdYlGn', ax=ax, linewidths=0.5, linecolor='gray')
    ax.set_title('Correlation Heatmap of Sales Features',
                 color='white', fontsize=14, fontweight='bold')
    ax.tick_params(colors='white')
    plt.tight_layout()
    plt.savefig('plot8_heatmap.png', dpi=150, bbox_inches='tight', facecolor=DARK)
    plt.show()
    print('✅ Plot 8 saved!')

## 1️⃣1️⃣ Conclusion

### 🔍 Key Findings:

1. **Branch Performance**: All three branches perform similarly, with slight variations in total revenue.

2. **Top Product Line**: Food and Beverages and Sports & Travel generate the highest revenue.

3. **Customer Type**: Members and Normal customers contribute almost equally to revenue.

4. **Payment**: Cash, Ewallet and Credit Card are used almost equally by customers.

5. **Peak Hours**: Supermarket sees highest sales between 10AM-2PM and 6PM-8PM.

6. **Ratings**: Average customer rating is around 7/10, indicating good satisfaction.

7. **Gender**: Male and Female customers contribute equally to overall revenue.

### 💡 Insights:
To boost sales, the supermarket should focus on **peak hour promotions**, improve **lower-rated product lines**, and offer **member loyalty rewards** to increase repeat purchases.

---
*Project completed as part of Data Science Internship*